# Welcome to MUSICA!

This tutorial will walk you through getting an installation working and running a box model.

## Installation

You probably arrived at this tutorial through [binder](https://mybinder.org/). If so, you won't need to follow this section as we've taken care of installing everything for you.

musica is available from [pypi](https://pypi.org/project/musica/). Below are instructions that will install musica's dependencies alongside all of the dependencies requires for this tutorial

```bash
pip install musica[tutorial]
```

We do recommend you install into some virtual environment. We've included instructions for both [`uv`](https://docs.astral.sh/uv/), and [`conda`]

### uv

```bash
uv python install 3.13
uv venv --python 3.13

source .venv/bin/activate   # macOS/Linux
.venv\Scripts\activate      # Windows

uv pip install musica[tutorial] # bash
uv pip install 'musica[tutorial]' # zsh
```

### Conda

```bash
conda create --name musica python=3.13
conda activate musica
pip install musica[tutorial] # bash
pip install 'musica[tutorial]' # zsh
```

### CLI

musica-python comes with a cli tool, `musica-cli`. This can be useful to copy out some of our examples, or printing the version. You can read more about it's usage and options [here](https://ncar-musica.readthedocs.io/en/latest/user_guide/cli.html). To verify your installation worked, you can ask it to print the version number

```bash
musica-cli --version
musica 0.14.5 (MICM 3.11.0, TUV-x 0.14.0, CARMA 4.0.0)
```

# Using musica

Let's start off by importing musica and getting the versions in python.

## Version checks

In [1]:
import musica

In [2]:
print(musica.__version__)

0.14.5


Each of musica's componenents can be imported as a separate package. Your system may or may not have access to tuvx and carma if fortran support was not included for the package installed on your system, so some of the cells below may fail.

In [3]:
print(musica.micm.__version__)

3.11.0


In [4]:
print(musica.tuvx.__version__)

0.14.0


In [5]:
print(musica.carma.__version__)

4.0.0


To run chemistry you only need `micm`, which is our ODE solver. `tuv-x` can provide photolysis rates and `carma` can provide aerosol functionality. There are separate tutorials for each of those so we will forget about them for now.

# A simple gas-phase system from a configuration

MUSICA can be configured using a valid [mechanism configuration](https://mechanismconfiguration.readthedocs.io/en/latest/v1/index.html) file. We've provided several packaged examples with musica to make their usage easy. We will run a simple 3-species system below. We'll start by importing the required parts of musica. 

- `musica`, this will give us access to all parts of musica, including `MICM`, our gas-phase solver
- `Parser`, which can read our mechanism files
- `find_config_path`, a utility function that will make it easy to load our example

In [6]:
import musica
from musica.mechanism_configuration import Parser
from musica.utils import find_config_path

In [15]:
parser = Parser()
mechanism = parser.parse(find_config_path("v1", "analytical", "config.json"))

In [16]:
for reaction in mechanism.reactions:
    print(f"[{reaction.type.name}] {reaction.to_equation()}")

[Arrhenius] A -> B
[Arrhenius] B -> C


Above, we had our parser read the mechanism file and return a valid mechanism. We can now use this to create a `MICM` object which can give us a state. These two together, `solver` and `state`, allow us to set concnetrations and environmental values and evolve the chemistry forward in time in one grid cell.

In [17]:
solver = musica.MICM(mechanism=mechanism)
state = solver.create_state()

There are more options you can pass to `MICM` to select the solver type. The `state` can also be configured to solve for multiple grid cells, which you'll read more about in the next tutorial.

For now, we can set the initial conditions and move the chemistry forward!

In [18]:
state.set_conditions(temperatures=[298.15], pressures=[101325])
state.set_concentrations(
    {
        'A' : [1.0],
        'B' : [0.0]
    }
)

Now all that's left is to solve and collect the concentrations as they change over time. For now, we'll print them out. Later tutorials have more code to make some pretty graphs.

`solver.solve` takes in a state and timestep to solve for. Let's solve for 10 seconds and see how things change

In [19]:
def print_concentrations(t, state):
    cs = state.get_concentrations()
    print(f"{t}\t| {cs['A'][0]:.4f}\t| {cs['B'][0]:.4f}")

print_concentrations(0, state)
for t in range(1, 11):
    solver.solve(state, 1)
    print_concentrations(t, state)

0	| 1.0000	| 0.0000
1	| 0.9960	| 0.0040
2	| 0.9920	| 0.0079
3	| 0.9881	| 0.0119
4	| 0.9841	| 0.0157
5	| 0.9802	| 0.0196
6	| 0.9763	| 0.0234
7	| 0.9724	| 0.0272
8	| 0.9685	| 0.0310
9	| 0.9646	| 0.0347
10	| 0.9608	| 0.0384


And that's your first MUSICA box model! Read on to learn how to define mechanisms in-code for solving and to do more some more complex tasks.